# Phase 13: demo assets for the Gradio interface (STEP N, optional, Version E)
The fusion models of Version E are fed by fine-tuned text and audio encoders whose weights were never kept (Phases 3b / 4b store only their out-of-fold predictions). A live demo needs real encoders, so this phase **trains one text encoder and one audio encoder on the FULL training split with exactly the recipes recorded in `text_ft_meta.json` / `audio_ft_meta.json`** (same model, context, learning rate, epochs, class weights; fixed epochs, nothing selected on dev or test) and saves them next to the fusion weights.
**What is saved** in `demo_assets/` (about 1.5 GB): `text_encoder.pt` + `text_tokenizer/` (fine-tuned RoBERTa, fp16), `audio_encoder.pt` (fine-tuned WavLM, fp16), the best fusion checkpoints (baseline and the validation-selected proposed model, every seed), the module files and config / scaler / z-score statistics the demo needs, and `demo_manifest.json`.
**Honest note.** The reported thesis numbers use out-of-fold predictions of five fold models; the demo uses one full-train encoder per modality, so its predictions differ slightly from the reported ones (this is the usual stacking situation at inference time). The video branch uses the frozen HSEmotion face embedding, exactly as in Version E.
**Safety.** The phase only reads pipeline outputs and writes to `demo_assets/`; no pipeline file, feature or result is modified. It is OPTIONAL in Main: if it fails, Phases 1-12 and their results are unaffected. Finished encoders are re-used (settings id match), so a re-run costs little.
**Switch off:** `MELD_DEMO_ASSETS=0`. Nothing in this notebook has been run on real data.


In [ ]:
# Required Imports
import os, sys, json, time, hashlib, math, shutil, glob
import numpy as np
import pandas as pd
print("Libraries imported successfully.")


In [ ]:
# 1. Configuration
BASE_DIR = os.environ.get("MELD_BASE_DIR", "/kaggle/working")
CONFIG_PATH = os.path.join(BASE_DIR, "config.json")
config = json.load(open(CONFIG_PATH))
FEATURES_DIR, RESULTS_DIR = config["paths"]["features_dir"], config["paths"]["results_dir"]
ENABLED = os.environ.get("MELD_DEMO_ASSETS", "1").strip().lower() in ("1", "true", "yes")
DEMO_DIR = os.path.join(BASE_DIR, "demo_assets"); os.makedirs(DEMO_DIR, exist_ok=True)
MANIFEST_PATH, VERIF_PATH = os.path.join(DEMO_DIR, "demo_manifest.json"), os.path.join(RESULTS_DIR, "demo_verification.json")

def write_json(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)

def jclean(x):
    return json.loads(json.dumps(x))

def find_notebook(name):
    roots = [d for d in (os.environ.get("MELD_NOTEBOOK_DIR"), BASE_DIR, os.getcwd(), "/kaggle/working") if d]
    for d in roots:
        if os.path.isfile(os.path.join(d, name)):
            return os.path.join(d, name)
    frontier = [("/kaggle/input", 0)] if os.path.isdir("/kaggle/input") else []
    while frontier:
        d, depth = frontier.pop(0)
        try:
            entries = list(os.scandir(d))
        except OSError:
            continue
        for e in entries:
            if e.is_file() and e.name == name:
                return e.path
        if depth < 3:
            frontier += [(e.path, depth + 1) for e in entries if e.is_dir(follow_symlinks=False)]
    raise FileNotFoundError(name + " not found next to this notebook")

if not ENABLED:
    write_json(MANIFEST_PATH, {"enabled": False}); write_json(VERIF_PATH, {"n_tests": 0, "n_failed": 0, "results": []})
    print("MELD_DEMO_ASSETS=0: nothing is built. Remaining cells are skipped.")
else:
    TXT_META = json.load(open(os.path.join(FEATURES_DIR, "text_ft_meta.json"))); AUD_META = json.load(open(os.path.join(FEATURES_DIR, "audio_ft_meta.json")))
    assert TXT_META.get("enabled") and AUD_META.get("enabled"), "the demo needs the fine-tuned text and audio sets of Version E (MELD_TEXT_FT / MELD_AUDIO_FT enabled)"
    print("text recipe:", TXT_META["settings"]); print("audio recipe:", AUD_META["settings"])


In [ ]:
# 2. Pure helpers (unit-tested without torch): locating the fusion checkpoints, the file list, sizes
def collect_fusion(run_index, selected_name, base_dir):
    """run_index: list of dicts (name, seed, run_dir). Returns [(name, seed, best.pt path)] for the baseline and the selected proposed model (every seed).
    A run_dir recorded on another machine is re-located under base_dir/checkpoints by name/seed."""
    out = []
    for r in run_index:
        if r["name"] not in ("baseline", selected_name):
            continue
        p = os.path.join(r["run_dir"], "best.pt")
        if not os.path.exists(p):
            cand = sorted(glob.glob(os.path.join(base_dir, "checkpoints", "**", r["name"], f"seed{r['seed']}", "best.pt"), recursive=True))
            p = cand[-1] if cand else None
        out.append((r["name"], int(r["seed"]), p))
    return out

def dir_size_mb(path):
    tot = 0
    for dp, _, fn in os.walk(path):
        tot += sum(os.path.getsize(os.path.join(dp, f)) for f in fn)
    return tot / 1e6

def agree(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return float((a.argmax(1) == b.argmax(1)).mean()), float(np.abs(a - b).max())
print("Helpers defined.")


In [ ]:
# 3. Text encoder: Phase 3b's own cells (identical tokenisation / context / model) are executed in an ISOLATED namespace, then ONE model is trained on all training rows
TEXT_TRAIN_SRC = r'''
def train_all_and_save(out_dir):
    tr = np.arange(len(labels["train"]))
    torch.manual_seed(SEED + 999); np.random.seed(SEED + 999)
    model = TextClassifier().to(device)
    w = torch.tensor(class_weights_np(labels["train"], C, WEIGHT_POWER), dtype=torch.float32, device=device)
    lossf = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * math.ceil(len(tr) / BATCH)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    ytr = torch.tensor(labels["train"], dtype=torch.long)
    for ep in range(EPOCHS):
        model.train(); rng = np.random.RandomState(SEED * 1000 + 990 + ep); perm = rng.permutation(tr); tot, n = 0.0, 0
        for i in tqdm(range(0, len(perm), BATCH), desc=f"full-train text epoch {ep + 1}/{EPOCHS}", leave=False):
            sel = perm[i:i + BATCH]; inp, att, pm = collate([ids["train"][j] for j in sel]); y = ytr[sel].to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(inp, att, pm)
            loss = lossf(logits.float(), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
            tot += float(loss) * len(sel); n += len(sel)
        print(f"  full-train text epoch {ep + 1}: train loss {tot / n:.4f}")
    dev_logits = predict(model, ids["dev"])
    sd = {k: (v.detach().half().cpu() if v.is_floating_point() else v.detach().cpu()) for k, v in model.state_dict().items()}
    path = os.path.join(out_dir, "text_encoder.pt"); tmp = path + ".tmp"
    torch.save({"state_dict": sd, "settings": SETTINGS, "settings_id": SETTINGS_ID, "emotions": EMOTIONS, "ft_model": FT_MODEL, "context_utterances": CTX_N}, tmp); os.replace(tmp, path)
    tok.save_pretrained(os.path.join(out_dir, "text_tokenizer"))
    del model, opt; torch.cuda.empty_cache()
    m2 = TextClassifier().to(device)                                  # reload from the SAVED file and compare (proves the file is usable)
    ck = torch.load(path, map_location="cpu", weights_only=False)
    m2.load_state_dict({k: (v.float() if v.is_floating_point() else v) for k, v in ck["state_dict"].items()})
    sub = list(range(min(300, len(ids["dev"]))))
    re_logits = predict(m2, [ids["dev"][j] for j in sub])
    agr, dmax = agree(dev_logits[sub], re_logits)
    del m2; torch.cuda.empty_cache()
    dev_acc = float((dev_logits.argmax(1) == labels["dev"]).mean())
    return {"dev_accuracy": dev_acc, "dev_macro_f1": macro_f1_np(labels["dev"], dev_logits.argmax(1), C), "reload_argmax_agreement": agr, "reload_max_abs_logit_diff": dmax,
            "n_params_m": round(sum(v.numel() for v in sd.values()) / 1e6, 1), "file_mb": round(os.path.getsize(path) / 1e6, 1)}
'''

def build_text_ns():
    for k, v in {"MELD_P3B_MODEL": TXT_META["settings"]["ft_model"], "MELD_P3B_LR": repr(float(TXT_META["settings"]["lr"])), "MELD_P3B_BATCH": str(TXT_META["settings"]["batch"]),
                 "MELD_P3B_EPOCHS": str(TXT_META["settings"]["epochs"]), "MELD_P3B_FOLDS": str(TXT_META["settings"]["folds"]), "MELD_TEXT_CTX": str(TXT_META["settings"]["context_utterances"]),
                 "MELD_P3B_WEIGHT_POWER": repr(float(TXT_META["settings"]["weight_power"])), "MELD_TEXT_FT": "1"}.items():
        os.environ[k] = v
    nbk = json.load(open(find_notebook("Phase_03b_Finetuned_Text_OOF.ipynb")))["cells"]
    ns = {"__name__": "p3b_ns"}
    for i in (1, 2, 3, 4, 5):
        exec(compile("".join(nbk[i]["source"]), f"<phase3b:cell{i}>", "exec"), ns)
    assert jclean(ns["SETTINGS"]) == TXT_META["settings"], "the recipe rebuilt from Phase 3b differs from text_ft_meta.json"
    ns["agree"] = agree
    exec(compile(TEXT_TRAIN_SRC, "<phase13:text_train>", "exec"), ns)
    return ns

TEXT_DONE = os.path.join(DEMO_DIR, "text_encoder_meta.json")
def encoder_done(path, settings_id, files):
    try:
        return json.load(open(path)).get("settings_id") == settings_id and all(os.path.exists(os.path.join(DEMO_DIR, f)) for f in files)
    except Exception:
        return False

if ENABLED:
    if encoder_done(TEXT_DONE, TXT_META["settings_id"], ["text_encoder.pt"]):
        TEXT_INFO = json.load(open(TEXT_DONE)); print("text encoder already saved with the same recipe -> skipped", {k: TEXT_INFO[k] for k in ("dev_accuracy", "file_mb")})
    else:
        t0 = time.time(); _ns = build_text_ns(); TEXT_INFO = _ns["train_all_and_save"](DEMO_DIR)
        TEXT_INFO.update({"settings_id": TXT_META["settings_id"], "settings": TXT_META["settings"], "minutes": round((time.time() - t0) / 60, 1)}); write_json(TEXT_DONE, TEXT_INFO)
        del _ns
        print("text encoder saved:", {k: v for k, v in TEXT_INFO.items() if k not in ("settings",)})


In [ ]:
# 4. Audio encoder: Phase 4b's own cells in an ISOLATED namespace, then ONE WavLM trained on all usable training clips
AUDIO_TRAIN_SRC = r'''
def train_all_and_save(out_dir):
    tr = np.flatnonzero(valid["train"])
    torch.manual_seed(SEED + 999); np.random.seed(SEED + 999)
    model = AudioClassifier().to(device)
    w = torch.tensor(class_weights_np(labels["train"][tr], C, WEIGHT_POWER), dtype=torch.float32, device=device)
    lossf = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
    steps = EPOCHS * math.ceil(len(tr) / BATCH)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    ytr = torch.tensor(labels["train"], dtype=torch.long)
    for ep in range(EPOCHS):
        model.train(); rng = np.random.RandomState(SEED * 1000 + 990 + ep); tot, n = 0.0, 0
        for sel in tqdm(bucket_batches(LENS["train"], tr, BATCH, rng), desc=f"full-train audio epoch {ep + 1}/{EPOCHS}", leave=False):
            wv, ln = to_batch("train", sel); y = ytr[sel].to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(wv, ln)
            loss = lossf(logits.float(), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
            tot += float(loss) * len(sel); n += len(sel)
        print(f"  full-train audio epoch {ep + 1}: train loss {tot / n:.4f}")
    dv = np.flatnonzero(valid["dev"]); dev_logits = predict(model, "dev", dv)
    sd = {k: (v.detach().half().cpu() if v.is_floating_point() else v.detach().cpu()) for k, v in model.state_dict().items()}
    path = os.path.join(out_dir, "audio_encoder.pt"); tmp = path + ".tmp"
    torch.save({"state_dict": sd, "settings": SETTINGS, "settings_id": SETTINGS_ID, "emotions": EMOTIONS, "model_id": MODEL_ID, "sample_rate": SR, "max_len": MAX_LEN}, tmp); os.replace(tmp, path)
    del model, opt; torch.cuda.empty_cache()
    m2 = AudioClassifier().to(device)
    ck = torch.load(path, map_location="cpu", weights_only=False)
    m2.load_state_dict({k: (v.float() if v.is_floating_point() else v) for k, v in ck["state_dict"].items()})
    re_logits = predict(m2, "dev", dv)                            # SAME rows in the SAME order -> the same length-bucketed batches (WavLM has no attention mask, so padding changes logits slightly)
    agr, dmax = agree(dev_logits, re_logits)
    re_acc = float((re_logits.argmax(1) == labels["dev"][dv]).mean())
    del m2; torch.cuda.empty_cache()
    dev_acc = float((dev_logits.argmax(1) == labels["dev"][dv]).mean())
    return {"dev_accuracy": dev_acc, "dev_macro_f1": macro_f1_np(labels["dev"][dv], dev_logits.argmax(1), C), "reload_argmax_agreement": agr, "reload_max_abs_logit_diff": dmax, "reload_dev_accuracy_gap": abs(re_acc - dev_acc),
            "n_params_m": round(sum(v.numel() for v in sd.values()) / 1e6, 1), "file_mb": round(os.path.getsize(path) / 1e6, 1)}


def reverify(out_dir, recorded_dev_accuracy):
    """Re-check an already saved audio encoder without retraining: reload the file, predict ALL usable dev clips, compare the accuracy with the one recorded at training time."""
    path = os.path.join(out_dir, "audio_encoder.pt")
    m2 = AudioClassifier().to(device)
    ck = torch.load(path, map_location="cpu", weights_only=False)
    m2.load_state_dict({k: (v.float() if v.is_floating_point() else v) for k, v in ck["state_dict"].items()})
    dv = np.flatnonzero(valid["dev"]); lg = predict(m2, "dev", dv)
    acc = float((lg.argmax(1) == labels["dev"][dv]).mean()); del m2; torch.cuda.empty_cache()
    return {"reload_full_dev_accuracy": acc, "reload_dev_accuracy_gap": abs(acc - recorded_dev_accuracy)}
'''

def build_audio_ns():
    s = AUD_META["settings"]
    for k, v in {"MELD_P4B_EPOCHS": str(s["epochs"]), "MELD_P4B_LR": repr(float(s["lr"])), "MELD_P4B_BATCH": str(s["batch"]), "MELD_P4B_MAXSEC": repr(float(s["max_sec"])),
                 "MELD_P4B_WEIGHT_POWER": repr(float(s["weight_power"])), "MELD_P3B_FOLDS": str(s["folds"]), "MELD_AUDIO_FT": "1"}.items():
        os.environ[k] = v
    nbk = json.load(open(find_notebook("Phase_04b_Finetuned_Audio_OOF.ipynb")))["cells"]
    ns = {"__name__": "p4b_ns"}
    for i in (1, 2, 3, 4, 5):
        exec(compile("".join(nbk[i]["source"]), f"<phase4b:cell{i}>", "exec"), ns)
    assert jclean(ns["SETTINGS"]) == AUD_META["settings"], "the recipe rebuilt from Phase 4b differs from audio_ft_meta.json"
    ns["agree"] = agree
    exec(compile(AUDIO_TRAIN_SRC, "<phase13:audio_train>", "exec"), ns)
    return ns

AUDIO_DONE = os.path.join(DEMO_DIR, "audio_encoder_meta.json")
if ENABLED:
    if encoder_done(AUDIO_DONE, AUD_META["settings_id"], ["audio_encoder.pt"]):
        AUDIO_INFO = json.load(open(AUDIO_DONE)); print("audio encoder already saved with the same recipe -> not retrained", {k: AUDIO_INFO[k] for k in ("dev_accuracy", "file_mb")})
        if "reload_dev_accuracy_gap" not in AUDIO_INFO:          # saved by an earlier version of this phase: re-check the file itself (no training)
            _ns = build_audio_ns(); AUDIO_INFO.update(_ns["reverify"](DEMO_DIR, AUDIO_INFO["dev_accuracy"])); write_json(AUDIO_DONE, AUDIO_INFO); del _ns
            print("saved audio encoder re-checked on all usable dev clips:", {k: AUDIO_INFO[k] for k in ("reload_full_dev_accuracy", "reload_dev_accuracy_gap")})
    else:
        t0 = time.time(); _ns = build_audio_ns(); AUDIO_INFO = _ns["train_all_and_save"](DEMO_DIR)
        AUDIO_INFO.update({"settings_id": AUD_META["settings_id"], "settings": AUD_META["settings"], "minutes": round((time.time() - t0) / 60, 1)}); write_json(AUDIO_DONE, AUDIO_INFO)
        del _ns
        print("audio encoder saved:", {k: v for k, v in AUDIO_INFO.items() if k not in ("settings",)})


In [ ]:
# 5. Fusion checkpoints (baseline + validation-selected proposed model, every seed) and everything the demo needs to rebuild the inputs
if ENABLED:
    run_index = json.load(open(os.path.join(RESULTS_DIR, "phase9_run_index.json")))
    sel = json.load(open(os.path.join(RESULTS_DIR, "phase9_selected_lambda.json")))
    SEL_NAME = sel["selected_experiment"]
    FUSION = collect_fusion(run_index, SEL_NAME, BASE_DIR)
    missing = [(n, s) for n, s, p in FUSION if p is None]
    assert not missing, f"best.pt not found for {missing}: the Phase 9 checkpoints are missing in this working directory (restore the full Version E output, or re-run Phase 9 with MELD_FORCE=Phase_09)"
    COPIED = []
    for n, s, p in FUSION:
        dst = os.path.join(DEMO_DIR, "fusion", n, f"seed{s}"); os.makedirs(dst, exist_ok=True)
        shutil.copy2(p, os.path.join(dst, "best.pt"))
        cfgp = os.path.join(os.path.dirname(p), "config.json")
        if os.path.exists(cfgp):
            shutil.copy2(cfgp, os.path.join(dst, "config.json"))
        COPIED.append({"name": n, "seed": s, "file": os.path.relpath(os.path.join(dst, "best.pt"), DEMO_DIR), "source": p})
    SUPPORT = ["config.json", "context_config.json", "model_config.json", "meld_context.py", "meld_models.py", "meld_train.py", "meld_eval.py",
               "results/phase9_selected_lambda.json", "results/phase9_config.json", "results/loss_sweep_selected.json",
               "features/text_ft_meta.json", "features/audio_ft_meta.json", "features/text_features_meta.json", "features/audio_features_meta.json", "features/video_features_meta.json",
               "features/audio_scaler.npz", "features/video_scaler.npz"]
    SUPPORT_COPIED, SUPPORT_MISSING = [], []
    for rel in SUPPORT:
        src = os.path.join(BASE_DIR, rel)
        if os.path.exists(src):
            dst = os.path.join(DEMO_DIR, "support", rel); os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy2(src, dst); SUPPORT_COPIED.append(rel)
        else:
            SUPPORT_MISSING.append(rel)
    print(f"fusion checkpoints copied: {len(COPIED)} ({SEL_NAME} + baseline) | support files copied: {len(SUPPORT_COPIED)} | not present: {SUPPORT_MISSING}")


In [ ]:
# 6. Verification (a failure marks the optional phase as failed; nothing in the pipeline is changed), then the manifest
if ENABLED:
    import torch
    res = []
    def check(name, cond, detail=""):
        res.append({"test": name, "passed": bool(cond), "detail": str(detail)}); print(("PASS  " if cond else "FAIL  ") + name + (f"  [{detail}]" if detail else ""))
    check("N1 text encoder: saved file reloads and reproduces its own predictions", TEXT_INFO["reload_argmax_agreement"] >= 0.99 and TEXT_INFO["reload_max_abs_logit_diff"] < 0.5, (TEXT_INFO["reload_argmax_agreement"], TEXT_INFO["reload_max_abs_logit_diff"]))
    check("N2 audio encoder: the saved file reloads and reproduces the dev accuracy recorded at training time (all usable dev clips, same batches; gap <= 0.01)", AUDIO_INFO["reload_dev_accuracy_gap"] <= 0.01, AUDIO_INFO["reload_dev_accuracy_gap"])
    check("N3 text encoder learned something and is not trivially perfect (dev accuracy 0.30 .. 0.90)", 0.30 < TEXT_INFO["dev_accuracy"] < 0.90, round(TEXT_INFO["dev_accuracy"], 4))
    check("N4 audio encoder learned something and is not trivially perfect (dev accuracy 0.25 .. 0.90)", 0.25 < AUDIO_INFO["dev_accuracy"] < 0.90, round(AUDIO_INFO["dev_accuracy"], 4))
    check("N5 the encoder recipes equal the recipes of the pipeline features", TEXT_INFO["settings"] == TXT_META["settings"] and AUDIO_INFO["settings"] == AUD_META["settings"])
    n_seeds = len({s for _, s, _ in FUSION})
    check("N6 a best.pt exists for baseline and the selected proposed model for every seed", len(COPIED) == 2 * n_seeds and all(os.path.getsize(os.path.join(DEMO_DIR, c["file"])) > 0 for c in COPIED), (len(COPIED), n_seeds))
    bad = []
    for c in COPIED:
        ck = torch.load(os.path.join(DEMO_DIR, c["file"]), map_location="cpu", weights_only=False)
        if "model" not in ck or "model_kwargs" not in ck:
            bad.append(c["file"])
    check("N7 every fusion checkpoint loads and carries its weights and model_kwargs", not bad, bad[:3])
    check("N8 the z-score statistics (7 dims) of both fine-tuned sets are present", len(TXT_META["z_mu"]) == 7 and len(TXT_META["z_sd"]) == 7 and len(AUD_META["z_mu"]) == 7 and len(AUD_META["z_sd"]) == 7)
    must = ["config.json", "meld_context.py", "meld_models.py", "results/phase9_selected_lambda.json", "features/text_ft_meta.json", "features/audio_ft_meta.json"]
    check("N9 the essential support files are present", all(m in SUPPORT_COPIED for m in must), [m for m in must if m not in SUPPORT_COPIED])
    total_mb = dir_size_mb(DEMO_DIR)
    check("N10 demo_assets is below 4 GB", total_mb < 4000, f"{total_mb:.0f} MB")
    failed = [r for r in res if not r["passed"]]
    write_json(VERIF_PATH, {"n_tests": len(res), "n_failed": len(failed), "results": res})
    assert not failed, f"{len(failed)} check(s) failed: {[r['test'] for r in failed]}"
    write_json(MANIFEST_PATH, {"enabled": True, "version": "E", "selected_proposed": SEL_NAME, "emotions": config["emotions"], "text_encoder": {k: v for k, v in TEXT_INFO.items() if k != "settings"},
                               "audio_encoder": {k: v for k, v in AUDIO_INFO.items() if k != "settings"}, "video": "frozen HSEmotion face embedding (as in Version E)", "fusion": COPIED,
                               "support_files": SUPPORT_COPIED, "total_mb": round(total_mb, 1),
                               "note": "one full-train encoder per modality; reported thesis numbers use out-of-fold predictions, so demo predictions differ slightly. WavLM runs without an attention mask: run the demo audio one clip at a time (batch of 1)."})
    print(f"\nAll {len(res)} checks passed. demo_assets: {total_mb:.0f} MB. Phase 13 completed successfully.")
